In [3]:
! git clone https://github.com/fssdfhsdfsdk/CUDA-by-Example-source-code-for-the-book-s-examples-.git

Cloning into 'CUDA-by-Example-source-code-for-the-book-s-examples-'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 119 (delta 0), reused 2 (delta 0), pack-reused 115 (from 1)
Receiving objects: 100% (119/119), 1.09 MiB | 17.20 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [6]:
%cd /content/CUDA-by-Example-source-code-for-the-book-s-examples- 

/content/CUDA-by-Example-source-code-for-the-book-s-examples-


In [11]:
! ls

appendix_a  chapter05  chapter09  hello_world.cu	      README.md
bin	    chapter06  chapter10  lib
chapter03   chapter07  chapter11  LICENSE
chapter04   chapter08  common	  Pearson_CUDA_BookCover.jpg


## hello_world.cu

In [12]:
%%writefile hello_world.cu
#include "common/book.h"

int main( void ) {
    printf( "Hello, World!\n" );
    return 0;
}

Overwriting hello_world.cu



```
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
```

这只是 NVIDIA 在提醒你：你当前的编译目标架构（默认可能是较旧的 `sm_50` 或 `sm_60`）在未来的版本中将不再支持。
如果你想**消除这个警告**，或者针对 Colab 的 GPU（通常是 T4, 算力 7.5）进行**优化**，可以在编译时指定架构：

1\. 针对 Colab T4 优化（推荐）
Google Colab 最常用的显卡是 Tesla T4，其算力是 **7.5**。指定该架构后警告会消失，且执行效率更高：

```
!nvcc -arch=sm_75 hello.cu -o hello && ./hello
```

2. 仅仅是想关掉警告

如果你不在乎架构版本，只想清净点，按照提示加上屏蔽参数：

```
!nvcc hello.cu -Wno-deprecated-gpu-targets -o hello && ./hello
```

In [15]:
! nvcc hello_world.cu -arch=sm_75 -o hello && ./hello

Hello, World!


In [ ]:
%%writefile simple_kernel.cu
#include <iostream>


__global__ void kernel(void) {
    
}

int main (void) {
    kernel<<<1,1>>>();
    printf( "Hello, World!\n" );
    return 0;
}

Overwriting simple_kernel.cu


In [21]:
! nvcc simple_kernel.cu -arch=sm_75 -o hello && ./hello

Hello, World!


## simple_kernel_params.cu

💡 核心要点

1.  **同步性**：`cudaMemcpy` 是**阻塞型**的。这意味着 CPU 会等待数据拷贝完成后，才会执行下一行代码（不需要手动调 `cudaDeviceSynchronize()`）。
2.  **错误检查**：建议用 `cudaError_t` 接收返回值，确保拷贝成功。
3.  **性能瓶颈**：PCIe 总线的传输速度远慢于显存带宽。在实际开发中，应尽量**减少** Device 与 Host 之间的拷贝次数。

In [29]:
%%writefile simple_kernel_params.cu
#include "common/book.h"

__global__ void add(int a, int b, int *c) {
    *c = a * b;
}

int main(void) {
    int c;
    int *dev_c;
    HANDLE_ERROR( cudaMalloc( (void **)&dev_c, sizeof(int)));

    add<<<1, 1>>>(2, 7, dev_c);

    HANDLE_ERROR( cudaMemcpy(&c, dev_c, sizeof(int), cudaMemcpyDeviceToHost));
    printf("the 2 * 7 is: %d\n", c);
    
    int count = 0;
    HANDLE_ERROR( cudaGetDeviceCount(&count) );
    printf("the gpu count: %d\n", count);
    
    return 0;
}


Overwriting simple_kernel_params.cu


In [30]:
! nvcc simple_kernel_params.cu -arch=sm_75 -o hello && ./hello

the 2 * 7 is: 14
the gpu count: 1


In [48]:
%%writefile simple_device_call.cu
#include "common/book.h"

__device__ int multiem( int a, int b ) {
    return a * b;
}

__global__ void multi( int a, int b, int *c ) {
    *c = multiem( a, b );
}

int main( void ) {
    int c;
    int *dev_c;
    HANDLE_ERROR( cudaMalloc( (void**)&dev_c, sizeof(int) ) );

    multi<<<1,1>>>( 3, 7, dev_c );

    HANDLE_ERROR( cudaMemcpy( &c, dev_c, sizeof(int),
                              cudaMemcpyDeviceToHost ) );
    printf( "3 * 7 = %d\n", c );
    HANDLE_ERROR( cudaFree( dev_c ) );

    return 0;
}

Overwriting simple_device_call.cu


In [49]:
! nvcc simple_device_call.cu -arch=sm_75 -o hello && ./hello

3 * 7 = 21


## enume_gpu.cu


The maximum amount of shared memory a single block may use in bytes
    - 单个线程块（thread block）可使用的共享内存最大值取决于分配给该会话的具体 GPU 型号

-   **静态分配上限**：所有架构的静态共享内存（`__shared__` 定义的固定大小数组）默认限制均为 **48 KB**。超过此限制会导致编译错误。
-   **动态分配与开启**：若要使用超过 48 KB 的内存，必须使用**动态共享内存**（`extern __shared__`），并调用 `cudaFuncSetAttribute` 明确申请更大配额。
-   **硬件保留**：通常会有 **1 KB** 的共享内存在某些架构中被系统预留，因此实际可分配给线程块的最大值往往略小于 Streaming Multiprocessor (SM) 的物理总量。

In [31]:
%%writefile enume_gpu.cu
#include "common/book.h"


int main( void ) {
    cudaDeviceProp  prop;

    int count;
    HANDLE_ERROR( cudaGetDeviceCount( &count ) );
    for (int i=0; i< count; i++) {
        HANDLE_ERROR( cudaGetDeviceProperties( &prop, i ) );
        printf( "   --- General Information for device %d ---\n", i );
        printf( "Name:  %s\n", prop.name );
        printf( "Compute capability:  %d.%d\n", prop.major, prop.minor );
        printf( "Clock rate:  %d\n", prop.clockRate );
        printf( "Device copy overlap:  " );
        if (prop.deviceOverlap)
            printf( "Enabled\n" );
        else
            printf( "Disabled\n");
        printf( "Kernel execution timeout :  " );
        if (prop.kernelExecTimeoutEnabled)
            printf( "Enabled\n" );
        else
            printf( "Disabled\n" );

        printf( "   --- Memory Information for device %d ---\n", i );
        // printf( "Total global mem:  %ld\n", prop.totalGlobalMem );
        printf( "Total global mem:  %zu\n", prop.totalGlobalMem );
        printf( "Total constant Mem:  %ld\n", prop.totalConstMem );
        printf( "Max mem pitch:  %ld\n", prop.memPitch );
        printf( "Texture Alignment:  %ld\n", prop.textureAlignment );

        printf( "   --- MP Information for device %d ---\n", i );
        printf( "Multiprocessor count:  %d\n",
                    prop.multiProcessorCount );
        printf( "Shared mem per mp:  %ld\n", prop.sharedMemPerBlock );
        printf( "Registers per mp:  %d\n", prop.regsPerBlock );
        printf( "Threads in warp:  %d\n", prop.warpSize );
        printf( "Max threads per block:  %d\n",
                    prop.maxThreadsPerBlock );
        printf( "Max thread dimensions:  (%d, %d, %d)\n",
                    prop.maxThreadsDim[0], prop.maxThreadsDim[1],
                    prop.maxThreadsDim[2] );
        printf( "Max grid dimensions:  (%d, %d, %d)\n",
                    prop.maxGridSize[0], prop.maxGridSize[1],
                    prop.maxGridSize[2] );
        printf( "\n" );
    }
}

Writing enume_gpu.cu


In [32]:
! nvcc enume_gpu.cu -arch=sm_75 -o hello && ./hello

   --- General Information for device 0 ---
Name:  Tesla T4
Compute capability:  7.5
Clock rate:  1590000
Device copy overlap:  Enabled
Kernel execution timeout :  Disabled
   --- Memory Information for device 0 ---
Total global mem:  15637086208
Total constant Mem:  65536
Max mem pitch:  2147483647
Texture Alignment:  512
   --- MP Information for device 0 ---
Multiprocessor count:  40
Shared mem per mp:  49152
Registers per mp:  65536
Threads in warp:  32
Max threads per block:  1024
Max thread dimensions:  (1024, 1024, 64)
Max grid dimensions:  (2147483647, 65535, 65535)



## set_gpu.cu

In [ ]:
%%writefile set_gpu.cu
#include "common/book.h"

int main( void ) {
    cudaDeviceProp  prop;
    int dev;

    HANDLE_ERROR( cudaGetDevice( &dev ) );
    printf( "ID of current CUDA device:  %d\n", dev );

    memset( &prop, 0, sizeof( cudaDeviceProp ) );
    prop.major = 7;
    prop.minor = 5;
    HANDLE_ERROR( cudaChooseDevice( &dev, &prop ) );
    printf( "ID of CUDA device closest to revision 7.5:  %d\n", dev );
    HANDLE_ERROR( cudaSetDevice( dev ) );

    prop.major = 8;
    prop.minor = 0;
    HANDLE_ERROR( cudaChooseDevice( &dev, &prop ) );
    printf( "ID of CUDA device closest to revision 8.0:  %d\n", dev );
    HANDLE_ERROR( cudaSetDevice( dev ) );
}

Overwriting set_gpu.cu


In [43]:
! nvcc set_gpu.cu -arch=sm_75 -o hello && ./hello

ID of current CUDA device:  0
ID of CUDA device closest to revision 7.5:  0
ID of CUDA device closest to revision 8.0:  0
